<a href="https://colab.research.google.com/github/abyanrizz/practicalstatisticfordatascientist/blob/main/Chapter_6_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#chapter 6
Recent advances in statistics have been devoted to developing more powerful auto‐
mated techniques for predictive modeling—both regression and classification. These
methods, like those discussed in the previous chapter, are supervised methods—they
are trained on data where outcomes are known and learn to predict outcomes in new
data. They fall under the umbrella of statistical machine learning and are distin‐
guished from classical statistical methods in that they are data-driven and do not seek
to impose linear or other overall structure on the data. The K-Nearest Neighbors
method, for example, is quite simple: classify a record in accordance with how similar
records are classified. The most successful and widely used techniques are based on
ensemble learning applied to decision trees. The basic idea of ensemble learning is to
use many models to form a prediction, as opposed to using just a single model. Deci‐
sion trees are a flexible and automatic technique to learn rules about the relationships
between predictor variables and outcome variables. It turns out that the combination

of ensemble learning with decision trees leads to some of the best performing off-the-
shelf predictive modeling techniques.

The development of many of the techniques in statistical machine learning can be
traced back to the statisticians Leo Breiman (see Figure 6-1) at the University of Cali‐
fornia at Berkeley and Jerry Friedman at Stanford University. Their work, along with
that of other researchers at Berkeley and Stanford, started with the development of
tree models in 1984. The subsequent development of ensemble methods of bagging
and boosting in the 1990s established the foundation of statistical machine learning.

237

1 This and subsequent sections in this chapter © 2020 Datastats, LLC, Peter Bruce, Andrew Bruce, and Peter
Gedeck; used with permission.
Figure 6-1. Leo Breiman, who was a professor of statistics at UC Berkeley, was at the
forefront of the development of many techniques in a data scientist’s toolkit today

Machine Learning Versus Statistics
In the context of predictive modeling, what is the difference
between machine learning and statistics? There is not a bright line
dividing the two disciplines. Machine learning tends to be focused
more on developing efficient algorithms that scale to large data in
order to optimize the predictive model. Statistics generally pays
more attention to the probabilistic theory and underlying structure
of the model. Bagging, and the random forest (see “Bagging and
the Random Forest” on page 259), grew up firmly in the statistics
camp. Boosting (see “Boosting” on page 270), on the other hand,
has been developed in both disciplines but receives more attention
on the machine learning side of the divide. Regardless of the his‐
tory, the promise of boosting ensures that it will thrive as a techni‐
que in both statistics and machine learning.

K-Nearest Neighbors
The idea behind K-Nearest Neighbors (KNN) is very simple.1

For each record to be

classified or predicted:
1. Find K records that have similar features (i.e., similar predictor values).
2. For classification, find out what the majority class is among those similar records
and assign that class to the new record.
3. For prediction (also called KNN regression), find the average among those similar
records, and predict that average for the new record.

238 | Chapter 6: Statistical Machine Learning

Key Terms for K-Nearest Neighbors

Neighbor
A record that has similar predictor values to another record.
Distance metrics
Measures that sum up in a single number how far one record is from another.
Standardization
Subtract the mean and divide by the standard deviation.
Synonym
Normalization
z-score
The value that results after standardization.
K
The number of neighbors considered in the nearest neighbor calculation.
KNN is one of the simpler prediction/classification techniques: there is no model to
be fit (as in regression). This doesn’t mean that using KNN is an automatic proce‐
dure. The prediction results depend on how the features are scaled, how similarity is
measured, and how big K is set. Also, all predictors must be in numeric form. We will
illustrate how to use the KNN method with a classification example.
A Small Example: Predicting Loan Default
Table 6-1 shows a few records of personal loan data from LendingClub. LendingClub
is a leader in peer-to-peer lending in which pools of investors make personal loans to
individuals. The goal of an analysis would be to predict the outcome of a new poten‐
tial loan: paid off versus default.
Table 6-1. A few records and columns for LendingClub loan data
Outcome Loan amount Income Purpose Years employed Home ownership State
Paid off 10000 79100 debt_consolidation 11 MORTGAGE NV
Paid off 9600 48000 moving 5 MORTGAGE TN
Paid off 18800 120036 debt_consolidation 11 MORTGAGE MD
Default 15250 232000 small_business 9 MORTGAGE CA
Paid off 17050 35000 debt_consolidation 4 RENT MD
Paid off 5500 43000 debt_consolidation 4 RENT KS

K-Nearest Neighbors | 239

2 For this example, we take the first row in the loan200 data set as the newloan and exclude it from the data set
for training.
Consider a very simple model with just two predictor variables: dti, which is the
ratio of debt payments (excluding mortgage) to income, and payment_inc_ratio,
which is the ratio of the loan payment to income. Both ratios are multiplied by 100.

Using a small set of 200 loans, loan200, with known binary outcomes (default or no-
default, specified in the predictor outcome200), and with K set to 20, the KNN esti‐

mate for a new loan to be predicted, newloan, with dti=22.5 and
payment_inc_ratio=9 can be calculated in R as follows:2

In [ ]:
newloan <- loan200[1, 2:3, drop=FALSE]
knn_pred <- knn(train=loan200[-1, 2:3], test=newloan, cl=loan200[-1, 1], k=20)
knn_pred == 'paid off'
[1] TRUE

In [ ]:
predictors = ['payment_inc_ratio', 'dti']
outcome = 'outcome'
newloan = loan200.loc[0:0, predictors]
X = loan200.loc[1:, predictors]
y = loan200.loc[1:, outcome]
knn = KNeighborsClassifier(n_neighbors=20)
knn.fit(X, y)
knn.predict(newloan)

Figure 6-2 gives a visual display of this example. The new loan to be predicted is the
cross in the middle. The squares (paid off) and circles (default) are the training data.
The large black circle shows the boundary of the nearest 20 points. In this case, 9
defaulted loans lie within the circle, as compared with 11 paid-off loans. Hence the
predicted outcome of the loan is paid off. Note that if we consider only three nearest
neighbors, the prediction would be that the loan defaults.

240 | Chapter 6: Statistical Machine Learning

Figure 6-2. KNN prediction of loan default using two variables: debt-to-income ratio
and loan-payment-to-income ratio

While the output of KNN for classification is typically a binary
decision, such as default or paid off in the loan data, KNN routines
usually offer the opportunity to output a probability (propensity)
between 0 and 1. The probability is based on the fraction of one
class in the K nearest neighbors. In the preceding example, this
probability of default would have been estimated at 9

20 , or 0.45.

Using a probability score lets you use classification rules other than
simple majority votes (probability of 0.5). This is especially impor‐
tant in problems with imbalanced classes; see “Strategies for Imbal‐
anced Data” on page 230. For example, if the goal is to identify
members of a rare class, the cutoff would typically be set below
50%. One common approach is to set the cutoff at the probability
of the rare event.
Distance Metrics
Similarity (nearness) is determined using a distance metric, which is a function that
measures how far two records (x1
, x2
, ..., xp
) and (u1
, u2
, ..., up
) are from one another.
The most popular distance metric between two vectors is Euclidean distance. To
K-Nearest Neighbors | 241

measure the Euclidean distance between two vectors, subtract one from the other,
square the differences, sum them, and take the square root:
x1
− u1
2
+ x2
− u2
2
+ ⋯ + xp
− up
2
.

Another common distance metric for numeric data is Manhattan distance:
|x1
− u1
| + |x2
− u2
| + ⋯ + |xp
− up
|

Euclidean distance corresponds to the straight-line distance between two points (e.g.,
as the crow flies). Manhattan distance is the distance between two points traversed in
a single direction at a time (e.g., traveling along rectangular city blocks). For this rea‐

son, Manhattan distance is a useful approximation if similarity is defined as point-to-
point travel time.

In measuring distance between two vectors, variables (features) that are measured
with comparatively large scale will dominate the measure. For example, for the loan
data, the distance would be almost solely a function of the income and loan amount
variables, which are measured in tens or hundreds of thousands. Ratio variables
would count for practically nothing in comparison. We address this problem by
standardizing the data; see “Standardization (Normalization, z-Scores)” on page 243.

Other Distance Metrics
There are numerous other metrics for measuring distance between
vectors. For numeric data, Mahalanobis distance is attractive since
it accounts for the correlation between two variables. This is useful
since if two variables are highly correlated, Mahalanobis will essen‐
tially treat these as a single variable in terms of distance. Euclidean
and Manhattan distance do not account for the correlation, effec‐
tively placing greater weight on the attribute that underlies those
features. Mahalanobis distance is the Euclidean distance between
the principal components (see “Principal Components Analysis”
on page 284). The downside of using Mahalanobis distance is
increased computational effort and complexity; it is computed
using the covariance matrix (see “Covariance Matrix” on page 202).

One Hot Encoder
The loan data in Table 6-1 includes several factor (string) variables. Most statistical
and machine learning models require this type of variable to be converted to a series
of binary dummy variables conveying the same information, as in Table 6-2. Instead
of a single variable denoting the home occupant status as “owns with a mortgage,”
242 | Chapter 6: Statistical Machine Learning

“owns with no mortgage,” “rents,” or “other,” we end up with four binary variables.
The first would be “owns with a mortgage—Y/N,” the second would be “owns with no
mortgage—Y/N,” and so on. This one predictor, home occupant status, thus yields a
vector with one 1 and three 0s that can be used in statistical and machine learning
algorithms. The phrase one hot encoding comes from digital circuit terminology,
where it describes circuit settings in which only one bit is allowed to be positive (hot).
Table 6-2. Representing home ownership factor data in Table 6-1 as a numeric dummy
variable
OWNS_WITH_MORTGAGE OWNS_WITHOUT_MORTGAGE OTHER RENT
1 0 0 0
1 0 0 0
1 0 0 0
1 0 0 0
0 0 0 1
0 0 0 1

In linear and logistic regression, one hot encoding causes problems
with multicollinearity; see “Multicollinearity” on page 172. In such
cases, one dummy is omitted (its value can be inferred from the
other values). This is not an issue with KNN and other methods
discussed in this book.

Standardization (Normalization, z-Scores)
In measurement, we are often not so much interested in “how much” but in “how dif‐
ferent from the average.” Standardization, also called normalization, puts all variables
on similar scales by subtracting the mean and dividing by the standard deviation; in
this way, we ensure that a variable does not overly influence a model simply due to
the scale of its original measurement:
z =
x − x
s
The result of this transformation is commonly referred to as a z-score. Measurements
are then stated in terms of “standard deviations away from the mean.”
Normalization in this statistical context is not to be confused with
database normalization, which is the removal of redundant data
and the verification of data dependencies.

K-Nearest Neighbors | 243

For KNN and a few other procedures (e.g., principal components analysis and clus‐
tering), it is essential to consider standardizing the data prior to applying the proce‐
dure. To illustrate this idea, KNN is applied to the loan data using dti and
payment_inc_ratio (see “A Small Example: Predicting Loan Default” on page 239)
plus two other variables: revol_bal, the total revolving credit available to the appli‐
cant in dollars, and revol_util, the percent of the credit being used. The new record
to be predicted is shown here:

In [ ]:
newloan
payment_inc_ratio dti revol_bal revol_util
1 2.3932 1 1687 9.4

The magnitude of revol_bal, which is in dollars, is much bigger than that of the
other variables. The knn function returns the index of the nearest neighbors as an
attribute nn.index, and this can be used to show the top-five closest rows in loan_df: